In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from delta.tables import DeltaTable

def write_to_gold(
    input_df,
    target_path,
    merge_condition,
    columns_to_update
):
    """
    Creates the Gold Delta data if it does not exist.
    Otherwise, merges new data into existing Gold data.
    """

    final_df = (
        input_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
    )

    if not DeltaTable.isDeltaTable(spark, target_path):
        (
            final_df.write
            .format("delta")
            .mode("overwrite")
            .save(target_path)
        )
        print("Gold Delta data created successfully.")

    else:
        delta_table = DeltaTable.forPath(spark, target_path)

        update_map = {
            column: f"s.`{column}`"
            for column in columns_to_update
            if column not in ["created_timestamp", "updated_timestamp"]
        }
        update_map["updated_timestamp"] = "s.updated_timestamp"

        (
            delta_table.alias("t")
            .merge(final_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_map)
            .whenNotMatchedInsertAll()
            .execute()
        )
        print("Gold Delta data merged successfully.")